# Spaceship Titanic Soft Voting Pipeline

**Author:** Alexej K.  
**Date:** 16. January 2026  
**Comp:** [Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic/rules)  
**Date:** Rank 423 with 80.64% Accuarcy

## Description

In this Notebook my focal point was the creation of a seemless Pipeline which handels all the nesseceray Prepocessing sequences. There are some important Dependacies between the Features which resulted in the creation of 3 Main Pipelines and some smaller ones.

- The fully fitted Pipeline has the following Structure:
  - One Preprocessing Pipeline to generate some Features
  - One ColumnTransformer for the Imputation and Encodings
  - One Postprocessing for the last Features
- The Winning Submission was a Soft Voting-Model with the following Structure:
  - XGBClassifier
  - LogisticRegression
  - LGBMClassifier
  - SVC(kernel="rbf", probability=True)
  - Soft: weights [2, 1, 2, 1]


### File and Data Field Descriptions

- **train.csv** - Personal records for about two-thirds (~8700) of the passengers, to be used as training data.
  - **PassengerId** - A unique Id for each passenger. Each Id takes the form gggg\*pp where \*\*\_gggg indicates a group the passenger is travelling**\* with and **_pp is their number within the group_\*\*. People in a group are often family members, but not always.
  - **HomePlanet** - The planet the passenger departed from, typically their planet of permanent residence.
  - **CryoSleep** - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
  - **Cabin** - The cabin number where the passenger is staying. Takes the form **_deck/num/side, where side can be either P for Port or S for Starboard_**.
  - **Destination** - The planet the passenger will be debarking to.
  - **Age** - The age of the passenger.
  - **VIP** - Whether the passenger has **_paid for special VIP service_** during the voyage.
  - **RoomService, FoodCourt, ShoppingMall, Spa, VRDeck** - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
  - **Name** - The first and last names of the passenger.
  - **Transported** - Whether the passenger was transported to another dimension. **This is the target**, the column you are trying to predict.
- **test.csv** - Personal records for the remaining one-third (~4300) of the passengers, to be used as test data. Your task is to predict the value of Transported for the passengers in this set.
- **sample_submission.csv** - A submission file in the correct format.
  - **PassengerId** - Id for each passenger in the test set.
  - **Transported** - The target. For each passenger, predict either True or False.


### 1. Import and Setup


In [ ]:
# data analysis and wrangling
import pandas as pd
import numpy as np

# machine learning
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron, SGDClassifier, LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import balanced_accuracy_score


SEED = 42
np.random.seed(SEED)

In [ ]:
df_train = pd.read_csv("../data/spaceship-titanic/train.csv")
df_test = pd.read_csv("../data/spaceship-titanic/test.csv")

y_train = df_train["Transported"]

df_train.drop(["Transported"], axis=1, inplace=True)

### 2. Pipeline


In [41]:
cat_cols = ["HomePlanet", "Destination", "VIP", "Cabin", "Name"]
num_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck", "Balance"]
oh_cols = ["HomePlanet", "Destination", "Side", "Deck"]
or_cols = ["AgeGroup"]
sc_cols = []

In [42]:
class CabinSeperator(BaseEstimator, TransformerMixin):
    def __init__(self, cabin_col="Cabin", deck_col="Deck", side_col="Side"):
        self.cabin_col = cabin_col
        self.deck_col = deck_col
        self.side_col = side_col

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X, y=None):
        X = X.copy()

        X[self.deck_col] = X[self.cabin_col].apply(lambda x: str(x).split("/")[0] if pd.notnull(x) else "M")
        X[self.side_col] = X[self.cabin_col].apply(lambda x: str(x).split("/")[-1] if pd.notnull(x) else "M")

        return X

In [43]:
class FillHelper(BaseEstimator, TransformerMixin):
    def __init__(self, deck_col="Deck", home_col="HomePlanet", dest_col="Destination", cryo_col="CryoSleep"):
        self.deck_col = deck_col
        self.home_col = home_col
        self.dest_col = dest_col
        self.cryo_col = cryo_col

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X, y=None):
        X = X.copy()

        fill_hp = X.groupby([self.deck_col])[self.home_col].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else "Earth"
        )
        fill_dest = X.groupby([self.deck_col])[self.dest_col].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else "TRAPPIST-1e"
        )
        fill_cs = X.groupby([self.deck_col])[self.cryo_col].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else False
        )
        X[self.home_col] = X[self.home_col].fillna(fill_hp)
        X[self.dest_col] = X[self.dest_col].fillna(fill_dest)
        X[self.cryo_col] = X[self.cryo_col].fillna(fill_cs)

        return X

In [44]:
class BalanceAdder(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        balance_col="Balance",
        room_col="RoomService",
        food_col="FoodCourt",
        shop_col="ShoppingMall",
        spa_col="Spa",
        vr_col="VRDeck",
    ):
        self.balance_col = balance_col
        self.room_col = room_col
        self.food_col = food_col
        self.shop_col = shop_col
        self.spa_col = spa_col
        self.vr_col = vr_col

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X, y=None):
        X = X.copy()

        cols = [self.room_col, self.food_col, self.shop_col, self.spa_col, self.vr_col]

        X[cols] = X[cols].fillna(X[cols].median())
        X[self.balance_col] = X[cols].sum(axis=1)

        return X

In [45]:
class FamilySizeAdder(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        name_col="Name",
        cabin_col="Cabin",
        passenger_col="PassengerId",
        family_col="FamilySize",
        alone_col="isAlone",
        lastName_col="LastName",
        groupId_col="GroupId",
    ):
        self.name_col = name_col
        self.cabin_col = cabin_col
        self.passenger_col = passenger_col
        self.family_col = family_col
        self.alone_col = alone_col
        self.lastName_col = lastName_col
        self.groupId_col = groupId_col

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X, y=None):
        X = X.copy()
        X[self.lastName_col] = X[self.name_col].apply(lambda x: str(x).split(" ")[-1] if pd.notnull(x) else x)
        X[self.groupId_col] = X[self.passenger_col].apply(lambda x: str(x).split("_")[0] if pd.notnull(x) else x)

        count_name = X.groupby([self.groupId_col, self.lastName_col])[self.passenger_col].transform("count")
        count_cabin = X.groupby([self.groupId_col, self.cabin_col])[self.passenger_col].transform("count")

        X[self.family_col] = np.maximum(count_name.fillna(0), count_cabin.fillna(0))
        X[self.alone_col] = (X[self.family_col] == 1).astype(int)

        return X

In [46]:
class AgeGroupAdder(BaseEstimator, TransformerMixin):
    def __init__(self, age_col="Age", agegroup_col="AgeGroup", balance_col="Balance", family_col="FamilySize"):
        self.age_col = age_col
        self.agegroup_col = agegroup_col
        self.balance_col = balance_col
        self.family_col = family_col

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X, y=None):
        X = X.copy()

        fill_age = X.groupby([self.balance_col, self.family_col])[self.age_col].transform("median")
        X[self.age_col] = X[self.age_col].fillna(fill_age)
        X[self.age_col] = X[self.age_col].fillna(X[self.age_col].median())

        X[self.agegroup_col] = pd.qcut(X[self.age_col], 5)
        return X.drop(columns=[self.age_col])

In [47]:
class PostProcessing(BaseEstimator, TransformerMixin):
    def __init__(self, age_group_col="AgeGroup", int_cols=None, drop_cols=None):
        self.age_group_col = age_group_col
        self.int_cols = int_cols if int_cols is not None else ["CryoSleep", "VIP", "GroupId"]
        self.drop_cols = (
            drop_cols
            if drop_cols is not None
            else ["LastName", "PassengerId", "Name", "Cabin", "Transported", "HomePlanet", "Destination"]
        )

    def fit(self, X, y=None):
        self.ordinal_encoder_ = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.ordinal_encoder_.fit(X[[self.age_group_col]])
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X, y=None):
        X = X.copy()

        X[self.age_group_col] = self.ordinal_encoder_.transform(X[[self.age_group_col]]).ravel()

        for col in self.int_cols:
            if col in X.columns:
                X[col] = pd.to_numeric(X[col], downcast="integer").astype(int)

        cols_to_drop = [col for col in self.drop_cols if col in X.columns]
        return X.drop(columns=cols_to_drop)

In [48]:
"""numeric_preprocessor = Pipeline(steps=[("num_imputer", SimpleImputer(strategy="median"))])
cat_preprocessor = Pipeline(
    steps=[
        ("cat_imputer", SimpleImputer(strategy="most_frequent")),
    ]
)
onehot_preprocessor = Pipeline(
    steps=[
        ("oneHot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
ordinal_preprocessor = Pipeline(
    steps=[
        ("ordinal", OrdinalEncoder()),
    ]
)
global_pre = Pipeline(
    steps=[
        ("cabin_features_sperated", CabinSeperator()),
        ("fill_homeplanet_cryosleep", FillHelper()),
        ("sum_of_expenses", BalanceAdder()),
    ]
)
global_post = Pipeline(
    steps=[
        ("familySize_groupId", FamilySizeAdder()),
        ("agegroup_features", AgeGroupAdder()),
        ("PostProcessing", PostProcessing()),
    ]
)
preprocessor = ColumnTransformer(
    [
        ("num_and_sc", numeric_preprocessor, num_cols),
        ("cat", cat_preprocessor, cat_cols),
        ("onehot", onehot_preprocessor, oh_cols),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")"""

'numeric_preprocessor = Pipeline(steps=[("num_imputer", SimpleImputer(strategy="median"))])\ncat_preprocessor = Pipeline(\n    steps=[\n        ("cat_imputer", SimpleImputer(strategy="most_frequent")),\n    ]\n)\nonehot_preprocessor = Pipeline(\n    steps=[\n        ("oneHot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),\n    ]\n)\nordinal_preprocessor = Pipeline(\n    steps=[\n        ("ordinal", OrdinalEncoder()),\n    ]\n)\nglobal_pre = Pipeline(\n    steps=[\n        ("cabin_features_sperated", CabinSeperator()),\n        ("fill_homeplanet_cryosleep", FillHelper()),\n        ("sum_of_expenses", BalanceAdder()),\n    ]\n)\nglobal_post = Pipeline(\n    steps=[\n        ("familySize_groupId", FamilySizeAdder()),\n        ("agegroup_features", AgeGroupAdder()),\n        ("PostProcessing", PostProcessing()),\n    ]\n)\npreprocessor = ColumnTransformer(\n    [\n        ("num_and_sc", numeric_preprocessor, num_cols),\n        ("cat", cat_preprocessor, cat_cols),\n        

- cat_cols = ["HomePlanet", "Destination", "VIP", "Cabin", "Name"]
- num_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
- oh_cols = ["HomePlanet", "Destination", "Side", "Deck"]
- or_cols = ["AgeGroup"]
- sc_cols = []


In [ ]:
def get_pipeline(clf):
    numeric_preprocessor = Pipeline(steps=[("num_imputer", SimpleImputer(strategy="median"))])

    cat_preprocessor = Pipeline(
        steps=[
            ("cat_imputer", SimpleImputer(strategy="most_frequent", fill_value="missing")),
        ]
    )

    onehot_preprocessor = Pipeline(
        steps=[
            ("oneHot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    global_pre = Pipeline(
        steps=[
            ("cabin_features_sperated", CabinSeperator()),
            ("fill_homeplanet_cryosleep", FillHelper()),
            ("sum_of_expenses", BalanceAdder()),
        ]
    )

    global_post = Pipeline(
        steps=[
            ("familySize_groupId", FamilySizeAdder()),
            ("agegroup_features", AgeGroupAdder()),
            ("PostProcessing", PostProcessing()),
        ]
    )

    preprocessor = ColumnTransformer(
        [
            ("num_and_sc", numeric_preprocessor, num_cols),
            ("cat", cat_preprocessor, cat_cols),
            ("onehot", onehot_preprocessor, oh_cols),
        ],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )
    preprocessor.set_output(transform="pandas")

    full_pipeline = Pipeline(
        [
            ("global_pre", global_pre),
            ("preprocessor", preprocessor),
            ("global_post", global_post),
            ("model", clf),
        ]
    )
    return full_pipeline

In [ ]:
"""from sklearn.utils.validation import check_is_fitted, NotFittedError

model_xgb = get_pipeline(XGBClassifier(random_state=SEED))
model_xgb.fit(df_train, y_train)

# Prüfe jeden Step
for name, step in model_xgb.named_steps.items():
    try:
        check_is_fitted(step)
        print(f"✅ {name}: fitted")
    except NotFittedError:
        print(f"❌ {name}: NOT fitted!")"""

✅ global_pre: fitted
✅ preprocessor: fitted
✅ global_post: fitted
✅ model: fitted


In [50]:
full_pipeline = get_pipeline(XGBClassifier(random_state=SEED))

In [51]:
full_pipeline

,steps,"[('global_pre', ...), ('preprocessor', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('cabin_features_sperated', ...), ('fill_homeplanet_cryosleep', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,cabin_col,'Cabin'
,deck_col,'Deck'
,side_col,'Side'


### 3. Model, Training, Prediction and Submission


In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings(action="ignore", category=ConvergenceWarning)

In [52]:
# Pipelien w/o (clf)
pre_only = Pipeline(full_pipeline.steps[:-1])

X_trans = pre_only.fit_transform(df_train, y_train)
print(type(X_trans), X_trans.shape)
X_trans.info()

<class 'pandas.core.frame.DataFrame'> (8693, 30)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   RoomService                8693 non-null   float64
 1   FoodCourt                  8693 non-null   float64
 2   ShoppingMall               8693 non-null   float64
 3   Spa                        8693 non-null   float64
 4   VRDeck                     8693 non-null   float64
 5   Balance                    8693 non-null   float64
 6   VIP                        8693 non-null   int32  
 7   HomePlanet_Earth           8693 non-null   float64
 8   HomePlanet_Europa          8693 non-null   float64
 9   HomePlanet_Mars            8693 non-null   float64
 10  Destination_55 Cancri e    8693 non-null   float64
 11  Destination_PSO J318.5-22  8693 non-null   float64
 12  Destination_TRAPPIST-1e    8693 non-null   float64
 13 

In [54]:
def custom_val_score(clf, score_func=balanced_accuracy_score):

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    scores = []

    for train_idx, val_idx in skf.split(df_train, df_train):
        X_t, X_v = df_train.iloc[train_idx], df_train.iloc[val_idx]
        y_t, y_v = df_train.iloc[train_idx], df_train.iloc[val_idx]

        clf.fit(X_t, y_t)
        scores.append(score_func(y_v, clf.predict(X_v)))

    return round(np.array(scores).mean() * 100, 3)

In [55]:
def simple_predict(model):

    model.fit(df_train, y_train)
    y_pred = model.predict(df_test)

    print("Modell: {}".format(str(model.__class__).split(".")[-1]))
    print(f"Accuracy: {round(model.score(df_train, y_train) * 100, 2)}")

    scores = cross_val_score(model, df_train, y_train, cv=5)
    print("Mean = {}, Scores : {}".format(round(np.mean(scores) * 100, 2), scores))
    return y_pred

In [ ]:
"""y_pred_logreg = simple_predict(get_pipeline(LogisticRegression(random_state=SEED)))
y_pred_svc = simple_predict(get_pipeline(SVC(random_state=SEED)))
y_pred_lsvc = simple_predict(get_pipeline(LinearSVC(random_state=SEED)))
y_pred_sgd = simple_predict(get_pipeline(SGDClassifier(random_state=SEED)))
y_pred_knn = simple_predict(get_pipeline(KNeighborsClassifier()))
y_pred_gaus = simple_predict(get_pipeline(GaussianNB()))
y_pred_perceptron = simple_predict(get_pipeline(Perceptron(random_state=SEED)))
y_pred_dtree = simple_predict(get_pipeline(DecisionTreeClassifier(random_state=SEED)))
y_pred_randomforest = simple_predict(get_pipeline(RandomForestClassifier(random_state=SEED)))
y_pred_xgb = simple_predict(get_pipeline(XGBClassifier(random_state=SEED)))"""

In [ ]:
"""skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores_debug = []
for i, (train_idx, val_idx) in enumerate(skf.split(df_train, y_train)):
    X_t, X_v = df_train.iloc[train_idx], df_train.iloc[val_idx]
    y_t, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
    try:
        fold_model = get_pipeline(XGBClassifier(random_state=SEED))
        fold_model.fit(X_t, y_t)
        score = accuracy_score(y_v, fold_model.predict(X_v))
        scores_debug.append(score)
        print(f"Fold {i+1}: {score}")
    except Exception as e:
        print(f"Fold {i+1}: Error {e}")
        scores_debug.append(np.nan)

print("Debug Mean = {}, Scores : {}".format(round(np.mean(scores_debug) * 100, 2), scores_debug))"""

In [57]:
xgb = get_pipeline(XGBClassifier(random_state=SEED))
# rdf = RandomForestClassifier(random_state=SEED)
lr = get_pipeline(LogisticRegression(random_state=SEED))
# knn = KNeighborsClassifier(n_neighbors=5)
lgbm = get_pipeline(LGBMClassifier(random_state=SEED))

svc = get_pipeline(SVC(kernel="rbf", probability=True, random_state=SEED))


# VotingClassifier
voting_clf = VotingClassifier(
    estimators=[("xgb", xgb), ("lr", lr), ("svc", svc), ("lgbm", lgbm)],
    voting="soft",  # 'soft' für Wahrscheinlichkeiten, 'hard' für Mehrheitsentscheid
    weights=[2, 1, 1, 2],
)


voting_clf.fit(df_train, y_train)
y_pred_voting = voting_clf.predict(df_test)


print(f"Accuracy: {round(voting_clf.score(df_train, y_train) * 100, 2)}")
scores = cross_val_score(voting_clf, df_train, y_train, cv=5)

print("Mean = {}, Scores : {}".format(round(np.mean(scores) * 100, 2), scores))

# XGB + LR + SVC (kernel="rbf", probability=True) = 84.21 acc , 80.50% submission -> soft
# XGB + RBF + LR + SVC (kernel="rbf", probability=True) = 90.64 acc ,cv = 77.10, 80.336% submission -> soft
# XGB + RBF + LR + SVC (kernel="rbf", probability=True) = 90.59 acc ,cv = 76.82, 80.126% submission -> hard

# XGB + LR + LGBM + SVC (kernel="rbf", probability=True) = 85.31 acc ,cv = 73.84, 80.57% submission -> soft
# XGB + LGBM + SVC (kernel="rbf", probability=True) = 87.2 acc ,cv = 70.30, 80.50% submission -> soft

# XGB + LR + LGBM + SVC (kernel="rbf", probability=True) = 87.91 acc ,cv = 70.71, 80.64% submission -> soft, weights [2, 1, 2, 1]
# Rank: 436

c:\Users\alexe\miniconda3\envs\homl3\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Das System kann die angegebene Datei nicht finden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\alexe\miniconda3\envs\homl3\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Users\alexe\miniconda3\envs\homl3\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Users\alexe\miniconda3\envs\homl3\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\alexe\miniconda3\envs\homl3\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProc

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 4378, number of negative: 4315
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000894 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1839
[LightGBM] [Info] Number of data points in the train set: 8693, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503624 -> initscore=0.014495
[LightGBM] [Info] Start training from score 0.014495
Accuracy: 87.69
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000882 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1839
[Light

In [ ]:
"""from sklearn.ensemble import StackingClassifier

xgb = get_pipeline(XGBClassifier(random_state=SEED))
# rdf = RandomForestClassifier(random_state=SEED)
# lr = LogisticRegression(random_state=SEED)
# knn = KNeighborsClassifier(n_neighbors=5)
lgbm = get_pipeline(LGBMClassifier(random_state=SEED))
svc = get_pipeline(SVC(kernel="rbf", probability=True, random_state=SEED))

stacking_clf = StackingClassifier(
    estimators=[
        ("xgb", xgb),
        (
            "lgbm",
            lgbm,
        ),
        ("svc", svc),
    ],
    final_estimator=LogisticRegression(),  # Meta-learner
    cv=5,
)

stacking_clf.fit(df_train, y_train)
y_pred_stacking = stacking_clf.predict(df_test)

print(f"Accuracy: {round(stacking_clf.score(df_train, y_train) * 100, 2)}")
scores = cross_val_score(stacking_clf, df_train, y_train, cv=5)

print("Mean = {}, Scores : {}".format(round(np.mean(scores) * 100, 2), scores))

# XGB + LGBM + SVC (kernel="rbf", probability=True), LR= 80.82 acc ,cv = 77.37, 79.822% submission"""

In [60]:
model_xgb = get_pipeline(XGBClassifier(random_state=SEED))
model_xgb = model_xgb.fit(df_train, y_train)
y_pred_xgb = model_xgb.predict(df_test)

acc_xgb = round(model_xgb.score(df_train, y_train) * 100, 2)
acc_xgb

# 86 Default -> Simple Imputer median + most frequent, OneHotEncoder, XGBClassifier

# 91.38 all -> submission = 80.009%
# 87.21 without Balance cols -> submission = 74.35%
# 88,48 without Sides -> submission = 79.14%

# 91.90 all, but better fill order -> submission = 80.032% # 1240th place
# 91.79 with FamilySizeGrouped w/o FamilySize-> submission = 79.68%
# 90.88 binned Balance w/o Balance -> submission = 79.78%
# 92.43 unbinned Deck -> submission = 80.406% # 765th place

92.01

In [61]:
submission = pd.DataFrame({"PassengerId": df_test["PassengerId"], "Transported": y_pred_xgb.astype(bool)})
# submission.to_csv("../data/output-submissions/spaceTitanic/PipelineSubXGB.csv", index=False)